In [ ]:
import pandas as pd

In [ ]:
data= pd.read_csv('IMDB_Dataset.csv')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

data['sentiment']=le.fit_transform(data['sentiment'])
data['sentiment']

In [ ]:
data.columns

In [ ]:
x= data['review'].values
y= data['sentiment'].values

In [ ]:
from sklearn.model_selection import train_test_split
# 30% of the data is used for testing and the remaining 70% is used for training.
# The random_state parameter is set to 42 to ensure reproducibility of the results, 
# meaning that the same split will be obtained each time the code is run.
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
# The maximum number of words to keep, based on word frequency. 
# Only the most common vocab_siz-1 words will be kept in the vocabulary.
vocab_siz= 10000

# The maximum length of sequences in reviews. 
# Sequences longer than this will be truncated, and shorter ones will be padded.
Max_len = 200 

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# oov_token='<OOV>' is used to specify a token that will be used to represent out-of-vocabulary words,
# which are words that are not present in the training data.
# This allows the model to handle unseen words during inference without throwing an error,
# and it can help improve the model's performance on new, unseen data.
tokenizer= Tokenizer(num_words=vocab_siz, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train)


# Convert the text data into sequences of integers, 
# where each integer represents a word in the vocabulary.
# The tokenizer.texts_to_sequences() method takes a list of texts as input and 
# converts each text into a sequence of integers based on the word index created during the fitting process.
x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad the sequences to ensure that they all have the same length, 
# which is necessary for input into the neural network.
# truncating='post' means that if a sequence is longer than Max_len, 
# it will be truncated at the end (post).
x_train_pad = pad_sequences(x_train_seq,maxlen=Max_len,padding='post',truncating='post')
x_test_pad = pad_sequences(x_test_seq,maxlen=Max_len,padding='post',truncating='post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,Flatten

model = Sequential()

# The first layer is an Embedding layer that takes the input dimension (vocab_siz),
# the output dimension (64), and the input length (Max_len).
model.add(Embedding(input_dim=vocab_siz,output_dim=64,input_length=Max_len))

# The Flatten layer is used to convert the 3D output of the Embedding layer into a 2D array,
# which can be fed into the Dense layers.
model.add(Flatten())

# The next layer is a Dense layer with 64 units and ReLU activation function,
# which helps the model learn complex patterns in the data.
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
# Train the model using the fit() method, which takes the training data (x_train_pad and y_train),
# the number of epochs to train for (10), 
# the batch size (32), and a validation split of 0.1 
# (10% of the training data will be used for validation).    
h= model.fit(
    x_train_pad,y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


In [ ]:
# less than 0.5 is classified as negative (0) and 
# greater than or equal to 0.5 is classified as positive (1).
prediction = model.predict(x_test_pad[:5])
print(prediction)

In [ ]:
# Evaluate the model on the test data (x_test_pad and y_test) using the evaluate() method,
# which returns the loss and accuracy of the model on the test set.
# The verbose parameter is set to 1 to display the progress of the evaluation.
loss, accuracy =model.evaluate(x_test_pad,y_test,verbose=1)
print("Accuracy : ",accuracy)
print("Test Loss: ",loss)

In [ ]:
# The predict() method is used to generate predictions for the input data (x_test_pad) 
# using the trained model. 
y_pred_prob= model.predict(x_test_pad)
y_pred= (y_pred_prob>0.5).astype(int)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'positive'],
            yticklabels=['negative', 'positive'])
plt.xlabel('predicted')
plt.ylabel('actual')
plt.title('confusion matrix')
plt.show()

In [ ]:
print(cm)

In [ ]:

# classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))